# Notebook 05 — End-to-end advisor scenario

**ATLAS: Aligned Three-Layer Architecture for Semantics**  
FSI (Financial Services Industry) Semantic Layer Workshop on AWS — Workshop 2

---

Walk the advisor scenario across both UIs.

This notebook traces the full cross-persona flow from signal detection to
conversational follow-up. By the end, you will have seen how the Consumer Banker
and Wealth Advisor workflows connect through the shared backbone — and you will
understand why the audit trail must span both personas.

In [ ]:
# WS2 notebook setup — installs only what this notebook needs into the kernel.
# Skips uv sync (which installs the full agent stack and takes minutes).
import sys, subprocess

# Only install packages not already provided by the SageMaker base image
pkgs = ['rdflib>=7.0.0', 'pyshacl>=0.25.0', 'SPARQLWrapper>=2.0.0']

subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '--quiet',
     '--disable-pip-version-check'] + pkgs,
    cwd='/tmp'
)
print('WS2 dependencies ready.')


## Key terms for this notebook

| Term | What it is |
|------|------------|
| **Cross-UI flow** | A workflow that begins in one UI (Wholesale) and completes in another (Wealth). The referral starts with the Consumer Banker and lands with the Wealth Advisor. The audit trail must span both. |
| **Routing decision** | The moment the referral-orchestrator sends a referral to the Wealth Advisor. This decision links the two UIs: it references the signal detected in the Wholesale UI and stamps the relationship `routedByWorkflow`, which drives the Wealth UI's "New — routed to you" banner. |
| **"New — routed to you" banner → take-on** | The Wealth UI flags a routed client while `routedByWorkflow && !takenOnAt` (a real state, not a timer). **Take on client** writes a real `atlas:takenOnAt` (the `takeOnClient` mutation) and the banner clears — parallel to coverage (it does not change `isActive`/`coverageStartDate`). |
| **Audit trail** | The PROV-O-attributed chain of events from signal detection through routing to the advisor's take-on. Must be queryable across both personas to satisfy compliance requirements. |
| **Conversational follow-up** | The Wealth Advisor asking the conversational-context-manager a question about the client. It is honestly **single-turn**: each question is answered independently (`priorTurns` is always 0 because AgentCore Memory is not wired). The referral context is visible on the screen, not carried in a multi-turn memory. |

## The full flow connects two personas through one backbone

The ATLAS advisor scenario is not contained within a single UI. It begins when
**Dana Brooks (the Consumer Banker)** finds a client with a coverage gap and a
qualifying *derived* signal — the customer **Rachel Kim**, flagged with a Large
Deposit Pattern and No Advisor Coverage. Dana reviews the signal in the Wholesale
UI, the referral-rationale-drafter produces a narrative explanation, she approves
it (the human-in-the-loop gate), and the referral-orchestrator routes the referral
to **Marcus Webb (the Wealth Advisor)**. At that point, the workflow crosses the
UI boundary.

### The route → banner → take-on → reset loop

What lands on Marcus's side is the workshop's signature demo loop, and it is a
*real* state transition end to end, not a scripted animation:

1. **Routed → "New — routed to you."** When Dana approves, the workflow assigns
   Marcus as Rachel's advisor and stamps the relationship `routedByWorkflow`. In
   the Wealth UI, Rachel now appears in Marcus's book flagged **"New — routed to
   you"** — shown precisely while `routedByWorkflow && !takenOnAt` (the workflow
   created it and Marcus has not yet accepted it). It is the advisor's inbox of
   governed handoffs, not a timer.
2. **Take on client → banner clears.** Marcus opens Rachel's Client 360 and clicks
   **Take on client**, which writes a *real* `atlas:takenOnAt` timestamp (the
   `takeOnClient` mutation). The banner clears because `!takenOnAt` is no longer
   true. This is **parallel** to coverage: it writes only `takenOnAt` and does not
   touch `isActive` or `coverageStartDate` — coverage was active at routing and
   stays exactly as it was (the no-harm design).
3. **Reset.** The workshop **Reset** (`resetDemoRoutings`, on Dana's dashboard)
   deletes the demo-created routed relationships and routing decisions — clearing
   the banner too, since it is driven by those relationships — and returns the
   graph to its seed state so the walk can be re-run cleanly.

From the Client 360 Marcus can also ask the conversational surface a follow-up
question (`converse` → conversational-context-manager). That surface is honestly
**single-turn**: each question is answered independently (`priorTurns` is always 0
because AgentCore Memory is not wired). It does not carry prior-turn context — the
referral context is *visible on the screen*, not loaded into a multi-turn memory.

The audit trail must span this entire flow. A compliance officer querying the graph
should be able to trace from the original signal detection (who detected it, when,
what evidence) through the rationale (who drafted it, who approved it) through the
routing decision (which orchestrator, which advisor) to the advisor's take-on. This
is why the routing decision creates a PROV-O-attributed AuditRecord that references
both Dana's signal and Marcus's handoff. The record links the two UIs in the graph.

This cross-UI flow is what validates Thesis 2 at the workflow level. It is not
enough for two UIs to render different views of the same data — they must also
participate in shared workflows where actions in one UI produce effects in the
other. The referral loop proves this: Dana's action (approve referral) creates
Marcus's state (a "new" client he then takes on). Same backbone, different personas,
connected workflow.

This notebook walks the full loop as one narrative. Dana's half is taught from the
Wholesale side in
[`../phase-1-referral/06_wholesale_ui.ipynb`](../phase-1-referral/06_wholesale_ui.ipynb);
Marcus's half (banner + take-on) from the Wealth side in
[`03_wealth_ui.ipynb`](./03_wealth_ui.ipynb); the canonical presenter script is
[`../../DEMO.md`](../../DEMO.md) and [`07_demo_runbook.ipynb`](./07_demo_runbook.ipynb).

In [ ]:
import sys
import os
import json
import uuid
from datetime import datetime, timezone

# Workshop 1's shared helpers.
sys.path.insert(0, "../../../agentic-semantic-layer/notebooks/shared")

from pathlib import Path

SPEC_DIR = "../../spec/04-aws-agent-registry"

print("Setup complete.")
print("This notebook simulates the cross-UI flow locally.")

In [ ]:
# Build cell 1 — Step 1: Consumer Banker detects signal.
#
# The wealth-signal-detector fires on a client with engagement decay.

signal_event = {
    "step": 1,
    "action": "signal_detected",
    "persona": "atlas-consumer-banker",
    "ui": "Wholesale UI",
    "agent": "wealth-signal-detector",
    "client_uri": "atlas:client-rachel-kim",
    "signal_type": "EngagementDecay",
    "evidence": {
        "baseline_sessions_per_month": 12,
        "recent_90d_sessions": 4,
        "decay_ratio": 0.111,
    },
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "audit_id": str(uuid.uuid4()),
}

print("Step 1: Signal detected in Wholesale UI")
print(json.dumps(signal_event, indent=2))

In [ ]:
# Build cell 2 — Steps 2-3: Draft rationale and route referral.

# Step 2: Rationale drafted (Dana Brooks, the Consumer Banker, approves — the gate)
rationale_event = {
    "step": 2,
    "action": "rationale_drafted",
    "persona": "atlas-consumer-banker",
    "agent": "referral-rationale-drafter",
    "client_uri": signal_event["client_uri"],
    "rationale": "Client shows significant engagement decay (decay ratio 0.111). "
                 "Historical baseline of 12 sessions/month dropped to 4 over 90 days. "
                 "Recommend wealth advisor outreach to re-engage.",
    "is_probabilistic": True,
    "requires_human_review": True,
    "approved_by": "dana.brooks",  # the Consumer Banker (the human-in-the-loop gate)
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "audit_id": str(uuid.uuid4()),
    "parent_audit_id": signal_event["audit_id"],
}

# Step 3: Routing decision — assigns Marcus Webb (the Wealth Advisor) + stamps routedByWorkflow
routing_event = {
    "step": 3,
    "action": "referral_routed",
    "persona": "atlas-consumer-banker",
    "agent": "referral-orchestrator",
    "client_uri": signal_event["client_uri"],
    "routed_to_persona": "atlas-wealth-advisor",
    "routed_to_advisor": "marcus.webb",     # Marcus Webb, the Wealth Advisor
    "routed_by_workflow": True,             # stamps atlas:routedByWorkflow → drives the banner
    "taken_on_at": None,                    # null until Marcus takes the client on
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "audit_id": str(uuid.uuid4()),
    "parent_audit_id": rationale_event["audit_id"],
}

print("Step 2: Rationale drafted and approved")
print(f"  Rationale: {rationale_event['rationale'][:80]}...")
print(f"  Approved by: {rationale_event['approved_by']} (Consumer Banker — the gate)")
print()
print("Step 3: Referral routed")
print(f"  Routed to: {routing_event['routed_to_advisor']} (Wealth Advisor)")
print(f"  routedByWorkflow: {routing_event['routed_by_workflow']}  takenOnAt: {routing_event['taken_on_at']}")
print("  → In the Wealth UI this shows as the 'New — routed to you' banner")
print("    (routedByWorkflow && !takenOnAt).")

In [ ]:
# Build cell 3 — Steps 4-7: Wealth Advisor sees the banner, takes the client on, follows up.

# Step 4: Marcus signs in to the Wealth UI and sees the "New — routed to you" banner.
notification_event = {
    "step": 4,
    "action": "new_routed_banner_shown",
    "persona": "atlas-wealth-advisor",
    "ui": "Wealth UI",
    "client_uri": signal_event["client_uri"],
    "signal_type": signal_event["signal_type"],
    "rationale_summary": rationale_event["rationale"][:100],
    # The banner condition: the workflow created the relationship AND it isn't taken on yet.
    "banner_shown": routing_event["routed_by_workflow"] and routing_event["taken_on_at"] is None,
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "audit_id": str(uuid.uuid4()),
    "parent_audit_id": routing_event["audit_id"],
}

# Step 5: Marcus clicks "Take on client" → writes a REAL atlas:takenOnAt → banner clears.
# Parallel to coverage: does NOT change isActive / coverageStartDate.
take_on_event = {
    "step": 5,
    "action": "take_on_client",
    "persona": "atlas-wealth-advisor",
    "ui": "Wealth UI",
    "agent": "takeOnClient",  # the mutation
    "client_uri": signal_event["client_uri"],
    "taken_on_at": datetime.now(timezone.utc).isoformat(),  # the real timestamp written
    "banner_cleared": True,    # because !takenOnAt is now false
    "coverage_unchanged": True,  # isActive / coverageStartDate untouched (the no-harm design)
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "audit_id": str(uuid.uuid4()),
    "parent_audit_id": notification_event["audit_id"],
}

# Step 6: Conversational follow-up — SINGLE-TURN (priorTurns is always 0; no memory carried).
conversation_event = {
    "step": 6,
    "action": "conversational_followup",
    "persona": "atlas-wealth-advisor",
    "ui": "Wealth UI",
    "agent": "conversational-context-manager",
    "question": "What is this client's current AUM and engagement trend?",
    "prior_turns": 0,  # single-turn: AgentCore Memory is not wired; each question stands alone
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "audit_id": str(uuid.uuid4()),
    "parent_audit_id": take_on_event["audit_id"],
}

print("Step 4: 'New — routed to you' banner shown in Wealth UI")
print(f"  banner_shown (routedByWorkflow && !takenOnAt): {notification_event['banner_shown']}")
print()
print("Step 5: Take on client")
print(f"  takenOnAt written: {take_on_event['taken_on_at'][:19]}  → banner_cleared: {take_on_event['banner_cleared']}")
print(f"  coverage_unchanged (isActive/coverageStartDate untouched): {take_on_event['coverage_unchanged']}")
print()
print("Step 6: Conversational follow-up (single-turn)")
print(f"  Question:   {conversation_event['question']}")
print(f"  priorTurns: {conversation_event['prior_turns']} (AgentCore Memory not wired)")

In [ ]:
# Build cell 4 — Assemble the full audit trail.
# The loop, in order: signal → rationale → route (banner appears) → banner shown →
# take-on (banner clears) → conversational follow-up. Every event references its parent.

audit_trail = [
    signal_event,
    rationale_event,
    routing_event,
    notification_event,
    take_on_event,
    conversation_event,
]

print("Full cross-UI audit trail:")
print("=" * 60)
for event in audit_trail:
    ui = event.get("ui", "—")
    print(f"  Step {event['step']}: [{ui:<12}] {event['action']}")
    print(f"           persona: {event['persona']}")
    print(f"           audit_id: {event['audit_id'][:8]}...")
    if event.get("parent_audit_id"):
        print(f"           parent:   {event['parent_audit_id'][:8]}...")
    print()

# Verify chain integrity
personas_in_trail = {e["persona"] for e in audit_trail}
uis_in_trail = {e.get("ui") for e in audit_trail if e.get("ui")}
print(f"Personas spanned: {sorted(personas_in_trail)}")
print(f"UIs spanned:      {sorted(uis_in_trail)}")

## Verification

Two properties must hold: the audit trail spans both personas (proving the
cross-UI flow is traceable), and the routing decision links to both UIs (proving
the backbone connects the two applications). If either fails, the end-to-end
scenario is incomplete.

In [ ]:
# Verification cell 1 — Audit trail spans both personas.

print("Verifying audit trail spans both personas...")
print()

personas_in_trail = {e["persona"] for e in audit_trail}
expected_personas = {"atlas-consumer-banker", "atlas-wealth-advisor"}

print(f"Personas in audit trail: {sorted(personas_in_trail)}")
print(f"Expected personas:       {sorted(expected_personas)}")
print()

# Verify parent chain is unbroken
audit_ids = {e["audit_id"] for e in audit_trail}
broken_links = []
for event in audit_trail:
    parent = event.get("parent_audit_id")
    if parent and parent not in audit_ids:
        broken_links.append(event["step"])

print(f"Broken parent links: {broken_links if broken_links else 'none'}")
print()

if not expected_personas.issubset(personas_in_trail):
    print("VERIFICATION FAILED: Audit trail does not span both personas.")
    print("The cross-UI flow must include events from both Consumer Banker")
    print("and Wealth Advisor to be compliance-complete.")

assert expected_personas.issubset(personas_in_trail), (
    "Audit trail must span both atlas-consumer-banker and atlas-wealth-advisor. "
    "Cross-UI traceability is required for compliance."
)
assert not broken_links, (
    f"Audit trail has broken parent links at steps: {broken_links}. "
    "Every event must reference its parent for chain integrity."
)

print("[PASS] Audit trail spans both personas with unbroken parent chain.")

In [ ]:
# Verification cell 2 — Routing decision links both UIs.

print("Verifying routing decision links both UIs...")
print()

# The routing event (step 3) should be traceable from both directions:
# - Forward: routing → banner shown (Wealth UI)
# - Backward: routing ← rationale ← signal (Wholesale UI)

routing = next(e for e in audit_trail if e["action"] == "referral_routed")
notification = next(e for e in audit_trail if e["action"] == "new_routed_banner_shown")

# Forward link: the banner-shown event references routing
forward_linked = notification["parent_audit_id"] == routing["audit_id"]
print(f"Forward link (routing → banner shown): {forward_linked}")

# Backward link: routing references rationale which references signal
rationale = next(e for e in audit_trail if e["action"] == "rationale_drafted")
signal = next(e for e in audit_trail if e["action"] == "signal_detected")

backward_linked = (
    routing["parent_audit_id"] == rationale["audit_id"]
    and rationale["parent_audit_id"] == signal["audit_id"]
)
print(f"Backward link (routing ← rationale ← signal): {backward_linked}")
print()

# Routing spans both UIs
routing_persona = routing["persona"]
notification_persona = notification["persona"]
spans_both = routing_persona != notification_persona
print(f"Routing persona:      {routing_persona}")
print(f"Banner-shown persona: {notification_persona}")
print(f"Spans both personas:  {spans_both}")
print()

# The banner truth: shown while routedByWorkflow && !takenOnAt, then cleared by take-on.
take_on = next(e for e in audit_trail if e["action"] == "take_on_client")
banner_then_cleared = notification["banner_shown"] is True and take_on["banner_cleared"] is True
print(f"Banner shown then cleared by take-on (real transition): {banner_then_cleared}")
print()

if not (forward_linked and backward_linked and spans_both and banner_then_cleared):
    print("VERIFICATION FAILED: Routing decision does not properly link both UIs.")
    print("The routing event must be the bridge between Wholesale and Wealth UI")
    print("audit trails, with parent references in both directions, and the banner")
    print("must be a real routedByWorkflow && !takenOnAt state cleared by take-on.")

assert forward_linked and backward_linked and spans_both and banner_then_cleared, (
    "Routing decision must link both UIs: forward to the banner-shown event (Wealth UI) "
    "and backward to signal (Wholesale UI); the banner must be a real state cleared by take-on."
)

print("[PASS] Routing decision correctly links both UIs.")
print("The cross-UI workflow — route → banner → take-on → clear — is fully traceable.")

## What just changed

You have walked the full advisor scenario across both UIs as one loop: signal
detection and approval in the Wholesale UI (Dana Brooks), routing across the UI
boundary, the **"New — routed to you"** banner in the Wealth UI (Marcus Webb),
**Take on client** (a real `atlas:takenOnAt` that clears the banner, parallel to
unchanged coverage), and a single-turn conversational follow-up. The audit trail
spans both personas with an unbroken parent chain, and the banner is a real
`routedByWorkflow && !takenOnAt` state cleared by a genuine take-on — not a timer.

This proves the two UIs are not independent applications — they are connected
through the shared backbone, with the routing decision as the bridge. A compliance
officer can trace any referral from detection through take-on across both personas.
To run the loop live, the canonical script is [`../../DEMO.md`](../../DEMO.md) and
the presenter runbook [`07_demo_runbook.ipynb`](./07_demo_runbook.ipynb); the
per-persona halves are in
[`../phase-1-referral/06_wholesale_ui.ipynb`](../phase-1-referral/06_wholesale_ui.ipynb)
(Dana) and [`03_wealth_ui.ipynb`](./03_wealth_ui.ipynb) (Marcus); the workshop
**Reset** (`resetDemoRoutings`) returns the graph to seed state between runs.

The final notebook runs the Phase 2 acceptance suite to confirm that all components
— agents, memory, JWT auth, cross-UI flow — work together as specified.